In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv() 
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Check your .env file")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
base_dir = Path.cwd()
docs_dir = base_dir / "acme_hr_docs"
docs_dir.mkdir(parents=True, exist_ok=True)

sample_docs = {
    "leave_policy.txt": """
Acme HR Policy: Leave and Time Off (v1.3)
- Annual leave: Full-time employees receive 25 days per year, accrued monthly.
- Carryover: Up to 5 unused days may be carried into the next year if approved by a manager by Dec 15.
- Sick leave: Employees should notify their manager as early as possible. A doctor's note is required for absences of 3+ consecutive working days.
- Parental leave: Eligible employees may take up to 16 weeks paid parental leave (policy requires 12 months service).
""",
    "remote_work.txt": """
Acme HR Policy: Remote Work (v2.1)
- Hybrid default: Employees are expected in the office 2 days/week unless their role is designated fully remote.
- Core collaboration hours: 10:00–16:00 local time.
- Equipment: Acme provides a laptop. Additional equipment reimbursement up to £250 with receipts.
- Data security: Company VPN required when using public Wi-Fi.
""",
    "benefits.txt": """
Acme HR Policy: Benefits Overview (v1.0)
- Private medical insurance: Available after probation (3 months).
- Pension: 5% employer contribution when employee contributes at least 3%.
- Learning budget: £1,000 per year for role-relevant training with manager approval.
""",
    "conduct.txt": """
Acme HR Policy: Code of Conduct (v3.0)
- Anti-harassment: Acme maintains zero tolerance for harassment or discrimination.
- Reporting: Concerns can be raised with your manager, HR, or anonymously via the ethics hotline.
- Confidentiality: Employees must protect confidential information and follow security policies.
""",
}

for filename, content in sample_docs.items():
    (docs_dir / filename).write_text(content.strip(), encoding="utf-8")

print(f"Created {len(sample_docs)} Acme HR docs in: {docs_dir}")


Created 4 Acme HR docs in: c:\RAG\acme_hr_docs


In [ ]:
loader = DirectoryLoader(
    str(docs_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50) 
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")


Loaded 4 documents
Created 7 chunks


In [ ]:
persist_directory = str(base_dir / "acme_chroma_store")

import shutil
p = Path(persist_directory)
if p.exists():
    shutil.rmtree(p)

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="acme_hr_rag",
)

print("Vector store vectors:", vectorstore._collection.count())
print("Persisted to:", persist_directory)


Vector store vectors: 7
Persisted to: c:\RAG\acme_chroma_store


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):

    formatted = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        name = Path(src).name
        formatted.append(f"[Source: {name}]\n{d.page_content}")
    return "\n\n---\n\n".join(formatted)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Acme's HR assistant. Answer the user's question using ONLY the provided context. "
     "If the answer is not in the context, say: 'I don't have that information in the HR policies provided.' "
     "Be concise and policy accurate."),
    ("human",
     "Question:\n{question}\n\nContext:\n{context}\n\nAnswer:")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | format_docs,
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain

{
  question: RunnablePassthrough(),
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000209D3CC7B60>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are Acme's HR assistant. Answer the user's question using ONLY the provided context. If the answer is not in the context, say: 'I don't have that information in the HR policies provided.' Be concise and policy accurate."), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Question:\n{question}\n\nContext:\n{context}\n\nAnswer:'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_toke

In [11]:
def ask_acme_hr(question: str):
    print("QUESTION:", question)
    answer = rag_chain.invoke(question)
    print("\nANSWER:\n", answer)

    # show sources
    docs = retriever.invoke(question)
    print("\nSOURCES:")
    for i, d in enumerate(docs, start=1):
        src = Path(d.metadata.get("source")).name
        preview = d.page_content[:220].replace("\n", " ")
        print(f"{i}. {src} — {preview}...")


In [12]:
ask_acme_hr("How many annual leave days do full-time employees get, and can I carry over unused days?")


QUESTION: How many annual leave days do full-time employees get, and can I carry over unused days?

ANSWER:
 Full-time employees receive 25 days of annual leave per year, accrued monthly. You can carry over up to 5 unused days into the next year if approved by a manager by December 15.

SOURCES:
1. leave_policy.txt — Acme HR Policy: Leave and Time Off (v1.3) - Annual leave: Full-time employees receive 25 days per year, accrued monthly. - Carryover: Up to 5 unused days may be carried into the next year if approved by a manager by Dec ...
2. leave_policy.txt — - Sick leave: Employees should notify their manager as early as possible. A doctor's note is required for absences of 3+ consecutive working days. - Parental leave: Eligible employees may take up to 16 weeks paid parenta...
3. remote_work.txt — Acme HR Policy: Remote Work (v2.1) - Hybrid default: Employees are expected in the office 2 days/week unless their role is designated fully remote. - Core collaboration hours: 10:00–16:00 lo

In [13]:
ask_acme_hr("Do we have a policy for sabbaticals?")


QUESTION: Do we have a policy for sabbaticals?

ANSWER:
 I don't have that information in the HR policies provided.

SOURCES:
1. leave_policy.txt — - Sick leave: Employees should notify their manager as early as possible. A doctor's note is required for absences of 3+ consecutive working days. - Parental leave: Eligible employees may take up to 16 weeks paid parenta...
2. leave_policy.txt — Acme HR Policy: Leave and Time Off (v1.3) - Annual leave: Full-time employees receive 25 days per year, accrued monthly. - Carryover: Up to 5 unused days may be carried into the next year if approved by a manager by Dec ...
3. benefits.txt — Acme HR Policy: Benefits Overview (v1.0) - Private medical insurance: Available after probation (3 months). - Pension: 5% employer contribution when employee contributes at least 3%. - Learning budget: £1,000 per year fo...
4. remote_work.txt — Acme HR Policy: Remote Work (v2.1) - Hybrid default: Employees are expected in the office 2 days/week unless their role